In [1]:
import os
import torch

os.environ["CUDA_VISIBLE_DEVICES"] = "1"   # ← change to your GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU   :", torch.cuda.get_device_name(0))

HOME     = os.path.expanduser("~")   # or set to '/nfsshare/users/P126003189'
MS_BYTES = 8192   # 8 KB SUARA2 segment size — used by SUARA2 notebook

Device: cuda
GPU   : NVIDIA H200


In [2]:
import random, time, copy, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Data, Batch
from torch_geometric.nn import SAGEConv

from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    average_precision_score, precision_recall_curve
)
from sklearn.preprocessing import label_binarize
from scipy.stats import friedmanchisquare
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

print("Imports OK")


Imports OK


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU   :", torch.cuda.get_device_name(0))
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB")


Device: cuda
GPU   : NVIDIA H200
Memory: 150.12 GB


In [4]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(42)
print("Seed fixed to 42")


Seed fixed to 42


In [5]:
# ── Hyperparameters (paper §5.2.4) ────────────────────────────────────────────
NUM_CLASSES  = 5
CLASS_NAMES  = ["colon_aca", "colon_n", "lung_aca", "lung_n", "lung_scc"]
# ^ Must match sorted order from GNN notebook class_to_idx

BATCH_SIZE   = 32
LR           = 0.01       # SGD learning rate (paper)
WEIGHT_DECAY = 1e-4
NUM_EPOCHS   = 100
PATIENCE     = 20         # early stopping patience

print("Config:")
print(f"  Classes    : {CLASS_NAMES}")
print(f"  Batch size : {BATCH_SIZE}")
print(f"  LR         : {LR}  |  WD: {WEIGHT_DECAY}")
print(f"  Epochs     : {NUM_EPOCHS}  |  Patience: {PATIENCE}")


Config:
  Classes    : ['colon_aca', 'colon_n', 'lung_aca', 'lung_n', 'lung_scc']
  Batch size : 32
  LR         : 0.01  |  WD: 0.0001
  Epochs     : 100  |  Patience: 20


In [6]:
from torch.serialization import add_safe_globals
add_safe_globals([np._core.multiarray.scalar])

cnn_features = torch.load("lc25000_cnn_features_v2.pt", map_location="cpu").float()
gnn_features = torch.load("lc25000_gnn_features_v2.pt", map_location="cpu").float()
labels       = torch.load("lc25000_labels_v2.pt",       map_location="cpu").long()

# Trim to 25000 if extraction produced 25001
for name, t in [("CNN", cnn_features), ("GNN", gnn_features), ("labels", labels)]:
    if t.shape[0] == 25001:
        print(f"Trimming {name} from 25001 → 25000")

if cnn_features.shape[0] == 25001: cnn_features = cnn_features[:25000]
if gnn_features.shape[0] == 25001: gnn_features = gnn_features[:25000]
if labels.shape[0]       == 25001: labels       = labels[:25000]

print(f"CNN features : {cnn_features.shape}")
print(f"GNN features : {gnn_features.shape}")
print(f"Labels       : {labels.shape}  unique={torch.unique(labels).tolist()}")
assert cnn_features.shape == gnn_features.shape, "Shape mismatch CNN vs GNN!"
assert cnn_features.shape[0] == labels.shape[0], "Sample count mismatch!"
print("Load checks passed.")


Trimming CNN from 25001 → 25000
CNN features : torch.Size([25000, 196, 1024])
GNN features : torch.Size([25000, 196, 1024])
Labels       : torch.Size([25000])  unique=[0, 1, 2, 3, 4]
Load checks passed.


In [7]:
# If GNN notebook was run with the .ipynb_checkpoints fix, labels are already
# clean [0..4]. This cell handles any residual corrupt label just in case.
counts = Counter(labels.numpy())
print("Raw class distribution:", dict(sorted(counts.items())))

if len(counts) > NUM_CLASSES:
    # Remove the corrupt class (fewest samples)
    corrupt = min(counts, key=counts.get)
    print(f"Removing corrupt label {corrupt} ({counts[corrupt]} sample(s))")
    mask         = labels != corrupt
    cnn_features = cnn_features[mask]
    gnn_features = gnn_features[mask]
    labels       = labels[mask]
    # Remap to 0-indexed
    unique     = sorted(torch.unique(labels).tolist())
    label_map  = {old: new for new, old in enumerate(unique)}
    labels     = torch.tensor([label_map[l.item()] for l in labels], dtype=torch.long)
    print(f"Remapped: {label_map}")
else:
    label_map = {i: i for i in range(NUM_CLASSES)}

TOTAL = len(labels)
print(f"\nFinal: {TOTAL} samples | labels {torch.unique(labels).tolist()}")
assert labels.min() == 0 and labels.max() == NUM_CLASSES - 1
print("Label range verified [0, 4]")


Raw class distribution: {np.int64(0): 5000, np.int64(1): 5000, np.int64(2): 5000, np.int64(3): 5000, np.int64(4): 5000}

Final: 25000 samples | labels [0, 1, 2, 3, 4]
Label range verified [0, 4]


In [8]:
from sklearn.model_selection import train_test_split

# Paper §5.2.4: 7:1.5:1.5 split
TRAIN_SIZE = int(TOTAL * 0.70)
VAL_SIZE   = int(TOTAL * 0.15)

all_idx    = np.arange(TOTAL)
all_labels = labels.numpy()

train_idx, temp_idx = train_test_split(
    all_idx, train_size=TRAIN_SIZE, stratify=all_labels, random_state=42
)
val_idx, test_idx = train_test_split(
    temp_idx, train_size=VAL_SIZE,
    stratify=all_labels[temp_idx], random_state=42
)

train_idx = torch.tensor(train_idx)
val_idx   = torch.tensor(val_idx)
test_idx  = torch.tensor(test_idx)

train_cnn, val_cnn, test_cnn = cnn_features[train_idx], cnn_features[val_idx], cnn_features[test_idx]
train_gnn, val_gnn, test_gnn = gnn_features[train_idx], gnn_features[val_idx], gnn_features[test_idx]
train_lbl, val_lbl, test_lbl = labels[train_idx],       labels[val_idx],       labels[test_idx]

print(f"Train: {len(train_lbl):,}  Val: {len(val_lbl):,}  Test: {len(test_lbl):,}")
print(f"Train dist: {Counter(train_lbl.numpy())}")
print(f"Val dist  : {Counter(val_lbl.numpy())}")
print(f"Test dist : {Counter(test_lbl.numpy())}")


Train: 17,500  Val: 3,750  Test: 3,750
Train dist: Counter({np.int64(0): 3500, np.int64(3): 3500, np.int64(2): 3500, np.int64(4): 3500, np.int64(1): 3500})
Val dist  : Counter({np.int64(0): 750, np.int64(1): 750, np.int64(4): 750, np.int64(2): 750, np.int64(3): 750})
Test dist : Counter({np.int64(0): 750, np.int64(1): 750, np.int64(4): 750, np.int64(2): 750, np.int64(3): 750})


In [9]:
class LC25000Dataset(Dataset):
    def __init__(self, cnn, gnn, lbl):
        self.cnn = cnn.float()
        self.gnn = gnn.float()
        self.lbl = lbl.long()
    def __len__(self): return len(self.lbl)
    def __getitem__(self, i): return self.cnn[i], self.gnn[i], self.lbl[i]

train_ds = LC25000Dataset(train_cnn, train_gnn, train_lbl)
val_ds   = LC25000Dataset(val_cnn,   val_gnn,   val_lbl)
test_ds  = LC25000Dataset(test_cnn,  test_gnn,  test_lbl)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f"Loaders: train={len(train_loader)} | val={len(val_loader)} | test={len(test_loader)} batches")


Loaders: train=547 | val=118 | test=118 batches


## Model Architecture
### Attention modules (CBAM-style, paper §4.2 Eq.12–14)

In [10]:
class ChannelAttention(nn.Module):
    """Paper Eq.13: Mc(F) = σ(MLP(AvgPool(F^c)) + MLP(MaxPool(F^c)))"""
    def __init__(self, in_ch, reduction=16):
        super().__init__()
        hidden = max(in_ch // reduction, 1)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(in_ch, hidden, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, in_ch, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return self.sigmoid(self.mlp(self.avg_pool(x)) + self.mlp(self.max_pool(x)))


class SpatialAttention(nn.Module):
    """Paper Eq.14: Ms(F) = σ(Conv[AvgPool(F^s); MaxPool(F^s)])"""
    def __init__(self, kernel_size=7):
        super().__init__()
        padding = 3 if kernel_size == 7 else 1
        self.conv    = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        mx, _ = torch.max(x, dim=1, keepdim=True)
        return self.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))

print("ChannelAttention and SpatialAttention defined.")


ChannelAttention and SpatialAttention defined.


In [11]:
class Bottleneck(nn.Module):
    """ResNet-style bottleneck used inside AFF fusion stage."""
    def __init__(self, in_ch, filters, stride=1, dropout=0.2):
        super().__init__()
        f1, f2, f3 = filters
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_ch, f1, 1, bias=False), nn.BatchNorm2d(f1), nn.ReLU(inplace=True))
        self.conv2 = nn.Sequential(
            nn.Conv2d(f1, f2, 3, padding=1, bias=False), nn.BatchNorm2d(f2), nn.ReLU(inplace=True))
        self.conv3 = nn.Sequential(
            nn.Conv2d(f2, f3, 1, bias=False), nn.BatchNorm2d(f3))
        self.drop  = nn.Dropout2d(p=dropout)
        self.down  = nn.Sequential(
            nn.Conv2d(in_ch, f3, 1, stride=stride, bias=False), nn.BatchNorm2d(f3))
        self.stride = stride
        self.relu   = nn.ReLU(inplace=True)
    def forward(self, x):
        identity = self.down(x)
        out = self.drop(self.conv3(self.conv2(self.conv1(x))))
        if self.stride > 1:
            out = F.avg_pool2d(out, self.stride)
        return self.relu(out + identity)

print("Bottleneck defined.")


Bottleneck defined.


In [12]:
class AFFModule(nn.Module):
    """
    Paper §4.2, Fig.7: Adaptive Feature Fusion module.
    F = μ·C + (1-μ)·G  (Eq.11)
    F' = Ms[Mc(F)·F]·[Mc(F)·F]  (Eq.12)
    Input: CNN tokens [B,196,1024] and GNN tokens [B,196,1024]
    Output: fused spatial map [B, 2048, 7, 7]
    """
    def __init__(self, initial_mu=0.5, reduction=16):
        super().__init__()
        # μ parameterized as logit for unconstrained optimization
        self.mu_raw = nn.Parameter(
            torch.tensor(math.log(initial_mu / (1.0 - initial_mu)), dtype=torch.float32)
        )
        # Branch normalization — prevents μ collapse due to scale mismatch
        self.cnn_norm = nn.LayerNorm([1024, 14, 14])
        self.gnn_norm = nn.LayerNorm([1024, 14, 14])

        self.ch_att  = ChannelAttention(1024, reduction)
        self.sp_att  = SpatialAttention(kernel_size=7)

        # AFF fusion stage: 1024 → 2048, with spatial downsampling (/2 → 7×7)
        self.fusion = nn.Sequential(
            Bottleneck(1024, [512, 512, 2048], stride=1, dropout=0.2),
            Bottleneck(2048, [512, 512, 2048], stride=1, dropout=0.2),
            Bottleneck(2048, [512, 512, 2048], stride=2, dropout=0.2),  # 14→7
        )

    def tokens_to_map(self, tokens):
        """[B, 196, 1024] → [B, 1024, 14, 14]"""
        B = tokens.shape[0]
        return tokens.view(B, 14, 14, 1024).permute(0, 3, 1, 2).contiguous()

    def forward(self, cnn_tokens, gnn_tokens):
        C = self.cnn_norm(self.tokens_to_map(cnn_tokens))   # [B, 1024, 14, 14]
        G = self.gnn_norm(self.tokens_to_map(gnn_tokens))

        mu = torch.sigmoid(self.mu_raw)          # constrained to (0,1)
        F  = mu * C + (1.0 - mu) * G            # Eq.11

        mc = self.ch_att(F)                      # Eq.13
        F1 = mc * F
        ms = self.sp_att(F1)                     # Eq.14
        F2 = ms * F1                             # Eq.12: F' = Ms[Mc(F)·F]·[Mc(F)·F]

        out = self.fusion(F2)                    # [B, 2048, 7, 7]
        return out, mu

print("AFFModule defined.")


AFFModule defined.


### Graph edge indices
Paper §4.3: after AFF, pixels at same spatial location across channels → graph nodes. Neighboring pixels → edges.

In [13]:
def build_grid_edge_index(h=7, w=7, device="cpu"):
    """
    Full 4-connected grid graph for the 7×7 fused feature map.
    49 nodes, edges between spatially adjacent nodes (up/down/left/right).
    Used by: AFFCNet baseline (global aggregation step).
    """
    edges = []
    nid = lambda r, c: r * w + c
    for r in range(h):
        for c in range(w):
            if r + 1 < h: edges += [[nid(r,c), nid(r+1,c)], [nid(r+1,c), nid(r,c)]]
            if c + 1 < w: edges += [[nid(r,c), nid(r,c+1)], [nid(r,c+1), nid(r,c)]]
    return torch.tensor(edges, dtype=torch.long).t().contiguous().to(device)


def build_local_edge_index(h=7, w=7, block_size=2, device="cpu"):
    """
    SUARA Step-1 edges: only within local 2×2 spatial blocks.
    Analogy: SUARA process-row sub-allreduce (local subset aggregation).
    49 nodes partitioned into ceil(7/2)×ceil(7/2)=16 non-overlapping blocks.
    """
    edges = []
    nid = lambda r, c: r * w + c
    for br in range(0, h, block_size):
        for bc in range(0, w, block_size):
            block = [(r, c)
                     for r in range(br, min(br + block_size, h))
                     for c in range(bc, min(bc + block_size, w))]
            for i in range(len(block)):
                for j in range(i + 1, len(block)):
                    u = nid(*block[i]); v = nid(*block[j])
                    edges += [[u, v], [v, u]]
    if not edges:
        return torch.zeros((2, 0), dtype=torch.long, device=device)
    return torch.tensor(edges, dtype=torch.long).t().contiguous().to(device)


# Pre-build all edge indices
grid_edge_index  = build_grid_edge_index(7, 7, device=device)   # baseline
local_edge_index = build_local_edge_index(7, 7, 2, device=device)  # SUARA step-1

print(f"Global grid edge index  : {grid_edge_index.shape}  (49-node full grid)")
print(f"Local block edge index  : {local_edge_index.shape}  (SUARA step-1 partitions)")


Global grid edge index  : torch.Size([2, 168])  (49-node full grid)
Local block edge index  : torch.Size([2, 120])  (SUARA step-1 partitions)


In [14]:
class GraphSAGEClassifier(nn.Module):
    """
    Paper §4.3, Fig.9: N-layer GraphSAGE on the fused graph.
    Input: fused feature map [B, 2048, 7, 7] → 49-node graph.
    Output: class logits [B, num_classes].
    """
    DIMS = {
        1: [2048, 512],
        3: [2048, 1024, 1024, 512],
        5: [2048, 1024, 1024, 1024, 1024, 512],
        7: [2048, 1024, 1024, 1024, 1024, 1024, 1024, 512],
    }
    def __init__(self, num_layers=3, num_classes=5, dropout=0.2):
        super().__init__()
        assert num_layers in self.DIMS, f"num_layers must be one of {list(self.DIMS)}"
        self.dropout = dropout
        dims = self.DIMS[num_layers]
        self.sage_layers = nn.ModuleList([SAGEConv(dims[i], dims[i+1]) for i in range(len(dims)-1)])
        self.bn_layers   = nn.ModuleList([nn.BatchNorm1d(dims[i+1]) for i in range(len(dims)-1)])
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Linear(dims[-1], num_classes)

    def _map_to_pyg(self, feat_map, edge_index):
        """[B, 2048, 7, 7] → PyG Batch of 49-node graphs"""
        B = feat_map.shape[0]
        nodes = feat_map.permute(0, 2, 3, 1).reshape(B, 49, 2048)
        return Batch.from_data_list([Data(x=nodes[i], edge_index=edge_index) for i in range(B)])

    def forward(self, feat_map, edge_index):
        B     = feat_map.shape[0]
        pyg_b = self._map_to_pyg(feat_map, edge_index)
        x, ei, batch = pyg_b.x, pyg_b.edge_index, pyg_b.batch
        for sage, bn in zip(self.sage_layers, self.bn_layers):
            x = F.relu(bn(sage(x, ei)))
            x = F.dropout(x, p=self.dropout, training=self.training)
        feat_dim = x.shape[-1]
        x = x.view(B, 7, 7, feat_dim).permute(0, 3, 1, 2)
        x = self.pool(x).flatten(1)
        return self.fc(x)

print("GraphSAGEClassifier defined  (supports layers: 1, 3, 5, 7)")


GraphSAGEClassifier defined  (supports layers: 1, 3, 5, 7)


In [15]:
class AFFCNet(nn.Module):
    """Full AFFC-Net: AFF module + GraphSAGE global aggregation."""
    def __init__(self, num_graphsage_layers=3, num_classes=5, initial_mu=0.5):
        super().__init__()
        self.aff       = AFFModule(initial_mu=initial_mu)
        self.graphsage = GraphSAGEClassifier(num_layers=num_graphsage_layers,
                                             num_classes=num_classes, dropout=0.2)
    def forward(self, cnn_tokens, gnn_tokens, edge_index):
        fused, mu = self.aff(cnn_tokens, gnn_tokens)
        return self.graphsage(fused, edge_index), mu

print("AFFCNet defined.")


AFFCNet defined.


### SUARA-Inspired variant
SUARA2 decomposes allreduce into **L=2 serial steps** over process subsets (rows then columns). Applied to GraphSAGE: Step 1 = local aggregation within spatial partitions (analogous to row-subset allreduce); Step 2 = global aggregation across the full grid (analogous to column-subset allreduce). Asymptotic benefit: O(√N) instead of O(N) neighborhood aggregation per node.

In [16]:
class GraphSAGE_SUARA2(nn.Module):
    """
    SUARA2-inspired 2-step hierarchical GraphSAGE aggregation.

    Step 1 (local): SAGEConv on local 2×2 block sub-graphs only.
                    Analogy: SUARA row-subset sub-allreduce.
    Step 2 (global): SAGEConv on full 7×7 grid graph.
                    Analogy: SUARA column-subset sub-allreduce.

    This reduces the effective neighborhood size per step from O(N) to O(√N),
    matching SUARA2's asymptotic speedup of O(√P) over flat allreduce.
    """
    def __init__(self, num_classes=5, dropout=0.2):
        super().__init__()
        self.dropout = dropout
        # Step 1: local aggregation within 2×2 blocks
        self.local_sage1 = SAGEConv(2048, 1024)
        self.local_bn1   = nn.BatchNorm1d(1024)
        self.local_sage2 = SAGEConv(1024, 1024)
        self.local_bn2   = nn.BatchNorm1d(1024)
        # Step 2: global aggregation across full grid
        self.global_sage = SAGEConv(1024, 512)
        self.global_bn   = nn.BatchNorm1d(512)
        # Readout
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Linear(512, num_classes)

    def _map_to_pyg(self, feat_map, edge_index):
        B = feat_map.shape[0]
        nodes = feat_map.permute(0, 2, 3, 1).reshape(B, 49, 2048)
        return Batch.from_data_list([Data(x=nodes[i], edge_index=edge_index) for i in range(B)])

    def forward(self, feat_map, local_ei, global_ei):
        B = feat_map.shape[0]
        # Build both PyG batches
        local_b  = self._map_to_pyg(feat_map, local_ei)
        global_b = self._map_to_pyg(feat_map, global_ei)

        x   = local_b.x
        l_ei = local_b.edge_index
        g_ei = global_b.edge_index

        # Step 1: local
        x = F.relu(self.local_bn1(self.local_sage1(x, l_ei)))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.local_bn2(self.local_sage2(x, l_ei)))
        x = F.dropout(x, p=self.dropout, training=self.training)

        # Step 2: global
        x = F.relu(self.global_bn(self.global_sage(x, g_ei)))

        # Readout
        x = x.view(B, 7, 7, 512).permute(0, 3, 1, 2)
        x = self.pool(x).flatten(1)
        return self.fc(x)


class AFFCNet_SUARA(nn.Module):
    """AFFC-Net with SUARA2-inspired hierarchical aggregation (novelty)."""
    def __init__(self, num_classes=5, initial_mu=0.5):
        super().__init__()
        self.aff       = AFFModule(initial_mu=initial_mu)
        self.graphsage = GraphSAGE_SUARA2(num_classes=num_classes, dropout=0.2)

    def forward(self, cnn_tokens, gnn_tokens, local_ei, global_ei):
        fused, mu = self.aff(cnn_tokens, gnn_tokens)
        logits    = self.graphsage(fused, local_ei, global_ei)
        return logits, mu

print("AFFCNet_SUARA (novelty) defined.")


AFFCNet_SUARA (novelty) defined.


### Ablation models
Match paper Table 1 ablation rows: AFFC-Net without GNN, without AFF.

In [17]:
class AFFCNetNoGNN(nn.Module):
    """Ablation: CNN branch only. GNN tokens zeroed out, mu fixed to 1."""
    def __init__(self, num_classes=5):
        super().__init__()
        self.aff       = AFFModule(initial_mu=0.5)
        self.graphsage = GraphSAGEClassifier(num_layers=3, num_classes=num_classes, dropout=0.2)
    def forward(self, cnn_tokens, gnn_tokens, edge_index):
        C  = self.aff.tokens_to_map(cnn_tokens)
        mc = self.aff.ch_att(C)
        F1 = mc * C
        ms = self.aff.sp_att(F1)
        F2 = ms * F1
        return self.graphsage(self.aff.fusion(F2), edge_index), torch.tensor(1.0)


class AFFCNetNoAFF(nn.Module):
    """Ablation: no attention — plain adaptive weighted sum only, no CBAM."""
    def __init__(self, num_classes=5):
        super().__init__()
        # FIX: logit(0.5) = 0.0  (original had broken formula)
        self.mu_raw = nn.Parameter(torch.tensor(0.0))
        self.cnn_norm = nn.LayerNorm([1024, 14, 14])
        self.gnn_norm = nn.LayerNorm([1024, 14, 14])
        self.fusion   = nn.Sequential(
            Bottleneck(1024, [512, 512, 2048], stride=1, dropout=0.2),
            Bottleneck(2048, [512, 512, 2048], stride=1, dropout=0.2),
            Bottleneck(2048, [512, 512, 2048], stride=2, dropout=0.2),
        )
        self.graphsage = GraphSAGEClassifier(num_layers=3, num_classes=num_classes, dropout=0.2)
    def _tok2map(self, t):
        return t.view(t.shape[0], 14, 14, 1024).permute(0, 3, 1, 2).contiguous()
    def forward(self, cnn_tokens, gnn_tokens, edge_index):
        C  = self.cnn_norm(self._tok2map(cnn_tokens))
        G  = self.gnn_norm(self._tok2map(gnn_tokens))
        mu = torch.sigmoid(self.mu_raw)
        F  = mu * C + (1.0 - mu) * G            # no attention
        return self.graphsage(self.fusion(F), edge_index), mu

print("Ablation models defined (NoGNN, NoAFF).")


Ablation models defined (NoGNN, NoAFF).


In [18]:
model_3layer = AFFCNet(num_graphsage_layers=3, num_classes=NUM_CLASSES).to(device)
model_5layer = AFFCNet(num_graphsage_layers=5, num_classes=NUM_CLASSES).to(device)
model_1layer = AFFCNet(num_graphsage_layers=1, num_classes=NUM_CLASSES).to(device)
model_7layer = AFFCNet(num_graphsage_layers=7, num_classes=NUM_CLASSES).to(device)
model_no_gnn = AFFCNetNoGNN(num_classes=NUM_CLASSES).to(device)
model_no_aff = AFFCNetNoAFF(num_classes=NUM_CLASSES).to(device)
model_suara  = AFFCNet_SUARA(num_classes=NUM_CLASSES).to(device)

for name, m in [("3-layer", model_3layer), ("5-layer", model_5layer),
                ("1-layer", model_1layer), ("7-layer", model_7layer),
                ("no-GNN",  model_no_gnn), ("no-AFF",  model_no_aff),
                ("SUARA",   model_suara)]:
    n_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"  {name:<10}: {n_params:>10,} params")

# Sanity check forward pass
with torch.no_grad():
    cb, gb, lb = next(iter(train_loader))
    cb, gb = cb.to(device), gb.to(device)
    out3, mu3 = model_3layer(cb, gb, grid_edge_index)
    outs, mus = model_suara(cb, gb, local_edge_index, grid_edge_index)
    print(f"\n3-layer forward: logits={out3.shape}  mu={mu3.item():.4f}")
    print(f"SUARA   forward: logits={outs.shape}  mu={mus.item():.4f}")
    assert out3.shape == (BATCH_SIZE, NUM_CLASSES)
print("All forward passes OK.")


  3-layer   : 31,645,800 params
  5-layer   : 35,846,248 params
  1-layer   : 26,396,776 params
  7-layer   : 40,046,696 params
  no-GNN    : 31,645,800 params
  no-AFF    : 31,514,630 params
  SUARA     : 31,645,800 params

3-layer forward: logits=torch.Size([32, 5])  mu=0.5000
SUARA   forward: logits=torch.Size([32, 5])  mu=0.5000
All forward passes OK.


In [19]:
# LC25000 is balanced — class weights ~equal; CrossEntropyLoss matches paper
def compute_class_weights(labels, num_classes):
    counts  = torch.bincount(labels, minlength=num_classes).float()
    weights = counts.sum() / (num_classes * counts.clamp(min=1.0))
    return weights

class_weights = compute_class_weights(train_lbl, NUM_CLASSES).to(device)
criterion     = nn.CrossEntropyLoss(weight=class_weights)
print("Class weights:", class_weights.cpu().numpy().round(4))


Class weights: [1. 1. 1. 1. 1.]


In [20]:
def compute_specificity(cm):
    specs = []
    for i in range(cm.shape[0]):
        tp = cm[i, i]; fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp; tn = cm.sum() - (tp + fn + fp)
        specs.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)
    return np.array(specs)

def evaluate_predictions(y_true, y_pred, y_prob):
    cm    = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))
    specs = compute_specificity(cm)
    return {
        "weighted_f1"      : f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "per_class_f1"     : f1_score(y_true, y_pred, average=None,       zero_division=0),
        "accuracy"         : (np.array(y_true) == np.array(y_pred)).mean(),
        "precision_w"      : precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall_w"         : recall_score(y_true,    y_pred, average="weighted", zero_division=0),
        "specificity_macro": specs.mean(),
        "specificity"      : specs,
        "confusion_matrix" : cm,
        "y_true": np.array(y_true), "y_pred": np.array(y_pred), "y_prob": np.array(y_prob),
        "report": classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0)
    }

print("Metrics helpers ready.")


Metrics helpers ready.


In [21]:
def train_epoch(model, loader, criterion, optimizer, edge_index,
                suara_local_ei=None, suara_global_ei=None, is_suara=False):
    model.train()
    total_loss = total_n = 0
    all_preds, all_true, all_probs = [], [], []
    t_data = t_fwd = t_bwd = 0.0
    t_start = time.time()

    for cnn_b, gnn_b, lbl_b in loader:
        t0 = time.time()
        cnn_b = cnn_b.to(device, non_blocking=True)
        gnn_b = gnn_b.to(device, non_blocking=True)
        lbl_b = lbl_b.to(device, non_blocking=True)
        t_data += time.time() - t0

        optimizer.zero_grad()
        t0 = time.time()
        if is_suara:
            logits, mu = model(cnn_b, gnn_b, suara_local_ei, suara_global_ei)
        else:
            logits, mu = model(cnn_b, gnn_b, edge_index)
        loss = criterion(logits, lbl_b)
        t_fwd += time.time() - t0

        t0 = time.time()
        loss.backward()
        optimizer.step()
        t_bwd += time.time() - t0

        bs = lbl_b.size(0)
        total_loss += loss.item() * bs; total_n += bs
        all_probs.append(F.softmax(logits, dim=1).detach().cpu().numpy())
        all_preds.append(logits.argmax(1).detach().cpu().numpy())
        all_true.append(lbl_b.cpu().numpy())

    epoch_time = time.time() - t_start
    m = evaluate_predictions(
        np.concatenate(all_true), np.concatenate(all_preds), np.concatenate(all_probs))
    m["loss"]       = total_loss / total_n
    m["mu"]         = mu.item()
    m["epoch_time"] = epoch_time
    m["throughput"] = total_n / epoch_time
    m["t_data"]     = t_data     # data loading time
    m["t_fwd"]      = t_fwd      # forward pass time
    m["t_bwd"]      = t_bwd      # backward pass time
    return m


@torch.no_grad()
def eval_epoch(model, loader, criterion, edge_index,
               suara_local_ei=None, suara_global_ei=None, is_suara=False):
    model.eval()
    total_loss = total_n = 0
    all_preds, all_true, all_probs = [], [], []
    t_start = time.time()

    for cnn_b, gnn_b, lbl_b in loader:
        cnn_b = cnn_b.to(device, non_blocking=True)
        gnn_b = gnn_b.to(device, non_blocking=True)
        lbl_b = lbl_b.to(device, non_blocking=True)
        if is_suara:
            logits, mu = model(cnn_b, gnn_b, suara_local_ei, suara_global_ei)
        else:
            logits, mu = model(cnn_b, gnn_b, edge_index)
        loss = criterion(logits, lbl_b)
        bs = lbl_b.size(0)
        total_loss += loss.item() * bs; total_n += bs
        all_probs.append(F.softmax(logits, dim=1).cpu().numpy())
        all_preds.append(logits.argmax(1).cpu().numpy())
        all_true.append(lbl_b.cpu().numpy())

    m = evaluate_predictions(
        np.concatenate(all_true), np.concatenate(all_preds), np.concatenate(all_probs))
    m["loss"]       = total_loss / total_n
    m["mu"]         = mu.item()
    m["eval_time"]  = time.time() - t_start   # added
    return m

print("train_epoch / eval_epoch ready (with full timing instrumentation).")

train_epoch / eval_epoch ready (with full timing instrumentation).


In [22]:
import os

def train_affcnet(model, model_name, save_path, checkpoint_path=None,
                  edge_index=None, is_suara=False,
                  suara_local_ei=None, suara_global_ei=None):

    if edge_index is None:
        edge_index = grid_edge_index

    optimizer = torch.optim.SGD(model.parameters(), lr=LR,
                                momentum=0.9, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=8, min_lr=1e-5)

    best_val_f1 = -1.0; best_state = None; patience_ctr = 0; start_epoch = 1
    history = {k: [] for k in ["epoch", "train_loss", "val_loss", "train_f1",
                                "val_f1", "train_acc", "val_acc", "mu",
                                "epoch_time", "throughput", "t_data", "t_fwd", "t_bwd"]}

    if checkpoint_path and os.path.exists(checkpoint_path):
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])
    best_val_f1 = ckpt["best_val_f1"]; patience_ctr = ckpt["patience_counter"]
    start_epoch = ckpt["epoch"] + 1;   history = ckpt["history"]

    # ✅ Backfill keys added after checkpoint was saved
    for key in ["throughput", "t_data", "t_fwd", "t_bwd"]:
        if key not in history:
            history[key] = []

    if os.path.exists(save_path):
        best_state = torch.load(save_path, map_location=device)
    print(f"Resumed {model_name} from epoch {start_epoch}  best F1={best_val_f1:.4f}")

    # ── wall-clock start ──────────────────────────────────────────────────────
    wall_start = time.time()

    for epoch in range(start_epoch, NUM_EPOCHS + 1):
        train_m = train_epoch(model, train_loader, criterion, optimizer,
                              edge_index, suara_local_ei, suara_global_ei, is_suara)
        val_m   = eval_epoch(model, val_loader, criterion,
                             edge_index, suara_local_ei, suara_global_ei, is_suara)
        scheduler.step(val_m["weighted_f1"])

        # Append to history
        history["epoch"].append(epoch)
        history["train_loss"].append(train_m["loss"])
        history["val_loss"].append(val_m["loss"])
        history["train_f1"].append(train_m["weighted_f1"])
        history["val_f1"].append(val_m["weighted_f1"])
        history["train_acc"].append(train_m["accuracy"])
        history["val_acc"].append(val_m["accuracy"])
        history["mu"].append(train_m["mu"])
        history["epoch_time"].append(train_m["epoch_time"])
        history["throughput"].append(train_m["throughput"])
        history["t_data"].append(train_m["t_data"])
        history["t_fwd"].append(train_m["t_fwd"])
        history["t_bwd"].append(train_m["t_bwd"])

        lr_now = optimizer.param_groups[0]["lr"]
        print(f"{model_name} | Ep {epoch:03d} | "
              f"Loss={train_m['loss']:.4f} | TrainF1={train_m['weighted_f1']:.4f} | "
              f"ValF1={val_m['weighted_f1']:.4f} | mu={train_m['mu']:.4f} | "
              f"lr={lr_now:.5f} | {train_m['epoch_time']:.1f}s "
              f"[fwd={train_m['t_fwd']:.1f}s bwd={train_m['t_bwd']:.1f}s "
              f"data={train_m['t_data']:.1f}s] | "
              f"{train_m['throughput']:.0f} img/s")

        if val_m["weighted_f1"] > best_val_f1:
            best_val_f1 = val_m["weighted_f1"]; patience_ctr = 0
            best_state  = copy.deepcopy(model.state_dict())
            torch.save(best_state, save_path)
            print(f"  ✓ Best saved. ValF1={best_val_f1:.4f}")
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f"  Early stop at epoch {epoch}")
                break

        if checkpoint_path:
            torch.save({
                "epoch": epoch, "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict(),
                "best_val_f1": best_val_f1,
                "patience_counter": patience_ctr,
                "history": history,
            }, checkpoint_path)

    # ── wall-clock summary ────────────────────────────────────────────────────
    total_wall = time.time() - wall_start
    n_epochs_run = len(history["epoch"]) - (start_epoch - 1)

    print(f"\n{'─'*60}")
    print(f"  {model_name} — Training Complete")
    print(f"{'─'*60}")
    print(f"  Total wall-clock time : {total_wall/60:.1f} min  ({total_wall:.1f}s)")
    print(f"  Epochs run            : {n_epochs_run}")
    print(f"  Avg epoch time        : {np.mean(history['epoch_time']):.2f}s")
    print(f"  Avg fwd pass time     : {np.mean(history['t_fwd']):.2f}s/epoch")
    print(f"  Avg bwd pass time     : {np.mean(history['t_bwd']):.2f}s/epoch")
    print(f"  Avg data loading time : {np.mean(history['t_data']):.2f}s/epoch")
    print(f"  Avg throughput        : {np.mean(history['throughput']):.0f} img/s")
    print(f"  Best val F1           : {best_val_f1:.4f}")
    print(f"{'─'*60}\n")

    model.load_state_dict(best_state)
    return model, history, best_val_f1

print("train_affcnet loop ready.")

IndentationError: expected an indented block after 'if' statement on line 20 (3269210225.py, line 21)

## Train AFFCNet-3Layer

In [ ]:
trained_3layer, history_3layer, best_f1_3layer = train_affcnet(
    model=model_3layer,
    model_name="AFFCNet-3Layer",
    save_path="affcnet_lc25000_3layer_best_ver2.pt",
    checkpoint_path="affcnet_lc25000_3layer_ckpt_ver2.pt",
    edge_index=grid_edge_index
)


## Train AFFCNet-5Layer

In [ ]:
trained_5layer, history_5layer, best_f1_5layer = train_affcnet(
    model=model_5layer,
    model_name="AFFCNet-5Layer",
    save_path="affcnet_lc25000_5layer_best_ver2.pt",
    checkpoint_path="affcnet_lc25000_5layer_ckpt_ver2.pt",
    edge_index=grid_edge_index
)


## Train ablation: No-GNN

In [ ]:
trained_nognn, history_nognn, best_f1_nognn = train_affcnet(
    model=model_no_gnn, model_name="AFFCNet-NoGNN",
    save_path="affcnet_lc25000_nognn_best_ver2.pt",
    checkpoint_path="affcnet_lc25000_nognn_ckpt_ver2.pt", edge_index=grid_edge_index
)


## Train ablation: No-AFF

In [ ]:
trained_noaff, history_noaff, best_f1_noaff = train_affcnet(
    model=model_no_aff, model_name="AFFCNet-NoAFF",
    save_path="affcnet_lc25000_noaff_best_ver2.pt",
    checkpoint_path="affcnet_lc25000_noaff_ckpt_ver2.pt", edge_index=grid_edge_index
)


## Train SUARA variant (novelty)

In [ ]:
trained_suara, history_suara, best_f1_suara = train_affcnet(
    model=model_suara, model_name="AFFCNet-SUARA",
    save_path="affcnet_lc25000_suara_best_ver2.pt",
    checkpoint_path="affcnet_lc25000_suara_ckpt_ver2.pt",
    is_suara=True,
    suara_local_ei=local_edge_index,
    suara_global_ei=grid_edge_index
)


In [ ]:
# ── Comprehensive Computational Time Comparison ───────────────────────────────
print("=" * 78)
print("  COMPUTATIONAL TIME ANALYSIS — AFFCNet LC25000")
print("=" * 78)

all_histories = {
    "AFFCNet-3Layer (baseline)" : history_3layer,
    "AFFCNet-5Layer"            : history_5layer,
    "AFFCNet-SUARA  (ours)"     : history_suara,
    "AFFCNet-NoGNN  (ablation)" : history_nognn,
    "AFFCNet-NoAFF  (ablation)" : history_noaff,
}

print(f"\n{'Model':<28} {'Epochs':>6} {'Total(min)':>11} {'Avg ep(s)':>10} "
      f"{'Fwd(s)':>8} {'Bwd(s)':>8} {'Data(s)':>8} {'img/s':>8}")
print("─" * 93)

baseline_avg = np.mean(history_3layer["epoch_time"])

for name, hist in all_histories.items():
    n_ep   = len(hist["epoch"])
    total  = sum(hist["epoch_time"]) / 60
    avg_ep = np.mean(hist["epoch_time"])
    avg_fwd  = np.mean(hist["t_fwd"])  if "t_fwd"  in hist and hist["t_fwd"]  else 0.0
    avg_bwd  = np.mean(hist["t_bwd"])  if "t_bwd"  in hist and hist["t_bwd"]  else 0.0
    avg_data = np.mean(hist["t_data"]) if "t_data" in hist and hist["t_data"] else 0.0
    tput   = np.mean(hist["throughput"])
    print(f"{name:<28} {n_ep:>6} {total:>10.1f}m {avg_ep:>9.1f}s "
          f"{avg_fwd:>7.1f}s {avg_bwd:>7.1f}s {avg_data:>7.1f}s {tput:>7.0f}")

print("─" * 93)

# SUARA speedup analysis
baseline_time = np.mean(history_3layer["epoch_time"])
suara_time    = np.mean(history_suara["epoch_time"])
speedup       = baseline_time / suara_time
time_saved    = (1 - suara_time / baseline_time) * 100

print(f"\n  SUARA vs Baseline:")
print(f"  Avg epoch time  — Baseline: {baseline_time:.2f}s  |  SUARA: {suara_time:.2f}s")
print(f"  Speedup         : {speedup:.3f}x")
print(f"  Time reduction  : {time_saved:.1f}%")
print(f"  Throughput gain : {np.mean(history_suara['throughput'])/np.mean(history_3layer['throughput']):.3f}x")
print(f"\n  Theoretical SUARA2 speedup (paper): O(√P)")
print(f"  Observed single-GPU speedup       : {speedup:.2f}x")
print(f"  Note: Full O(√P) gain requires multi-node distributed setting (P=1024 → 32x)")
print("=" * 78)

## Resume helper (run if kernel crashes)

In [ ]:
# Re-instantiate and reload from checkpoints if kernel died
for model_obj, ckpt, name in [
    (model_3layer, "affcnet_lc25000_3layer_ckpt_ver2.pt",  "3-layer"),
    (model_5layer, "affcnet_lc25000_5layer_ckpt_ver2.pt",  "5-layer"),
    
    (model_no_gnn, "affcnet_lc25000_nognn_ckpt_ver2.pt",   "no-gnn"),
    (model_no_aff, "affcnet_lc25000_noaff_ckpt_ver2.pt",   "no-aff"),
    (model_suara,  "affcnet_lc25000_suara_ckpt_ver2.pt",   "suara"),
]:
    if os.path.exists(ckpt):
        c = torch.load(ckpt, map_location=device)
        model_obj.load_state_dict(c["model_state"])
        print(f"{name}: ep={c['epoch']}  best_F1={c['best_val_f1']:.4f}")
    else:
        print(f"{name}: no checkpoint")


## Test set evaluation — all models

In [ ]:
print("Evaluating all models on test set...")

test_3layer = eval_epoch(trained_3layer, test_loader, criterion, grid_edge_index)
test_5layer = eval_epoch(trained_5layer, test_loader, criterion, grid_edge_index)

test_nognn  = eval_epoch(trained_nognn,  test_loader, criterion, grid_edge_index)
test_noaff  = eval_epoch(trained_noaff,  test_loader, criterion, grid_edge_index)
test_suara  = eval_epoch(trained_suara,  test_loader, criterion, grid_edge_index,
                         suara_local_ei=local_edge_index,
                         suara_global_ei=grid_edge_index, is_suara=True)

print("All test evaluations complete.")


## Paper Table 2 reproduction — LC25000 metrics

In [ ]:
print("=" * 70)
print(f"{'Model':<28} {'Prec':>7} {'Recall':>7} {'Spec':>7} {'Acc':>7} {'F1':>7}")
print("-" * 70)
rows = [
    ("AFFC-NoGNN (ablation)",   test_nognn),
    ("AFFC-NoAFF (ablation)",   test_noaff),
    
    ("AFFCNet-3Layer (paper)",  test_3layer),
    ("AFFCNet-5Layer",          test_5layer),
    
    #("AFFCNet-SUARA (ours)",    test_suara),
]
for name, m in rows:
    print(f"{name:<28} "
          f"{m['precision_w']*100:>6.2f}% "
          f"{m['recall_w']*100:>6.2f}% "
          f"{m['specificity_macro']*100:>6.2f}% "
          f"{m['accuracy']*100:>6.2f}% "
          f"{m['weighted_f1']*100:>6.2f}%")
print("-" * 70)
print(f"{'Paper target':<28} {'99.84%':>7} {'99.84%':>7} {'99.96%':>7} {'99.84%':>7} {'99.84%':>7}")


## SUARA Efficiency Comparison
Accuracy + training speed side by side — the key novelty comparison table.

In [ ]:
print("=" * 95)
print(f"{'Model':<28} {'F1':>7} {'Acc':>7} {'Prec':>7} {'Recall':>7} "
      f"{'avg ep(s)':>10} {'img/s':>8} {'speedup':>9}")
print("─" * 95)

baseline_time = np.mean(history_3layer["epoch_time"])
baseline_tput = np.mean(history_3layer["throughput"])

comparison = [
    ("AFFCNet-3Layer (baseline)", test_3layer, history_3layer),
    ("AFFCNet-SUARA  (ours)",     test_suara,  history_suara),
    ("AFFCNet-5Layer",            test_5layer, history_5layer),
]
for name, tm, hist in comparison:
    avg_t   = np.mean(hist["epoch_time"])
    avg_s   = np.mean(hist["throughput"])
    speedup = baseline_time / avg_t
    print(f"{name:<28} "
          f"{tm['weighted_f1']*100:>6.2f}% "
          f"{tm['accuracy']*100:>6.2f}% "
          f"{tm['precision_w']*100:>6.2f}% "
          f"{tm['recall_w']*100:>6.2f}% "
          f"{avg_t:>9.1f}s "
          f"{avg_s:>7.0f}  "
          f"{speedup:>8.2f}x")

print("─" * 95)
print(f"\n  Key insight: SUARA-inspired hierarchical aggregation")
print(f"  reduces graph neighbourhood per step from O(N)→O(√N).")
print(f"  Analogous to SUARA2 process decomposition: 2 steps of √P")
print(f"  instead of 1 step of P.")
print(f"\n  Single-GPU speedup        : {baseline_time/np.mean(history_suara['epoch_time']):.2f}x")
print(f"  SUARA2 paper (P=1024)     : 2.00x average, 2.65x peak")
print(f"  Training time reduction   : {(1-np.mean(history_suara['epoch_time'])/baseline_time)*100:.1f}% (single GPU)")
print(f"  Paper reported reduction  : 9% (P=1024 nodes)")

## Ablation summary table

In [ ]:
print(f"\n{'Model':<30} {'Weighted F1':>12} {'Accuracy':>10}")
print("-" * 55)
for name, tm in [
    ("AFFCNet without GNN",    test_nognn),
    ("AFFCNet without AFF",    test_noaff),
    ("AFFCNet 3-layer (full)", test_3layer),
    ("AFFCNet 5-layer",        test_5layer),
    ("AFFCNet + SUARA (ours)", test_suara),
]:
    print(f"{name:<30} {tm['weighted_f1']*100:>11.2f}%  {tm['accuracy']*100:>9.2f}%")
print(f"\nPaper reported: F1=99.84%, Accuracy=99.84%")


## Per-class F1 report

In [ ]:
for model_name_str, tm in [
    ("3-layer", test_3layer), ("5-layer", test_5layer), ("SUARA", test_suara)
]:
    print(f"\n── Per-class F1 ({model_name_str}) ──────────────────────")
    for cname, f1v in zip(CLASS_NAMES, tm["per_class_f1"]):
        print(f"  {cname:<15}: {f1v*100:.2f}%")
    print(f"  Weighted F1  : {tm['weighted_f1']*100:.2f}%  (paper: 99.84%)")
    print(tm["report"])


## Friedman statistical significance test
Paper §5.3.2 tests statistical significance across GraphSAGE layer counts.

In [ ]:
from scipy.stats import wilcoxon

stat, p_value = wilcoxon(
    test_3layer["per_class_f1"],
    test_5layer["per_class_f1"]
)
print(f"Wilcoxon signed-rank test (3-layer vs 5-layer):")
print(f"  statistic : {stat:.4f}")
print(f"  p-value   : {p_value:.4f}")

## Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, hist, name in [
    (axes[0], history_3layer, "3-layer"),
    (axes[1], history_5layer, "5-layer"),
    (axes[2], history_suara,  "SUARA"),
]:
    ax.plot(hist["epoch"], hist["mu"], color="#1a73e8", lw=2, label="learned μ")
    ax.axhline(0.5,  color="gray",   linestyle="--", alpha=0.6, label="init μ=0.5")
    ax.axhline(0.88, color="orange", linestyle=":",  alpha=0.8, label="paper μ≈0.88")
    ax.set_title(f"μ evolution — {name}")
    ax.set_xlabel("Epoch"); ax.set_ylabel("μ (CNN weight)")
    ax.set_ylim(0, 1); ax.legend(); ax.grid(True, linestyle="--", alpha=0.4)
plt.suptitle("LC25000 — Adaptive Scaling Factor μ (CNN dominance)", fontsize=13)
plt.tight_layout()
plt.savefig("lc25000_mu_evolution-copy.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Learning curves — loss and F1
fig, axes = plt.subplots(3, 2, figsize=(14, 14))
for row, (hist, name) in enumerate([
    (history_3layer, "AFFCNet 3-layer (baseline)"),
    (history_5layer, "AFFCNet 5-layer"),
    (history_suara,  "AFFCNet SUARA (novelty)"),
]):
    axes[row, 0].plot(hist["epoch"], hist["train_loss"], label="Train")
    axes[row, 0].plot(hist["epoch"], hist["val_loss"],   label="Val")
    axes[row, 0].set_title(f"{name} — Loss")
    axes[row, 0].set_xlabel("Epoch"); axes[row, 0].legend()
    axes[row, 0].grid(True, linestyle="--", alpha=0.4)

    axes[row, 1].plot(hist["epoch"], hist["train_f1"], label="Train F1")
    axes[row, 1].plot(hist["epoch"], hist["val_f1"],   label="Val F1")
    axes[row, 1].set_title(f"{name} — Weighted F1")
    axes[row, 1].set_xlabel("Epoch"); axes[row, 1].legend()
    axes[row, 1].grid(True, linestyle="--", alpha=0.4)

plt.suptitle("LC25000 Learning Curves", fontsize=14)
plt.tight_layout()
plt.savefig("lc25000_learning_curves-copy.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Fig.11b reproduction — F1 and Accuracy vs GraphSAGE layer count
# Only include the layer variants you actually trained
layer_counts = [3, 5]
f1_vals  = [test_3layer["weighted_f1"]*100,
            test_5layer["weighted_f1"]*100]
acc_vals = [test_3layer["accuracy"]*100,
            test_5layer["accuracy"]*100]

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(layer_counts))   # [0, 1] — matches f1_vals and acc_vals
w = 0.35

bars1 = ax.bar(x - w/2, f1_vals,  w, label="Weighted F1", color="#1a73e8", alpha=0.85)
bars2 = ax.bar(x + w/2, acc_vals, w, label="Accuracy",    color="#e8711a", alpha=0.85)

for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels([f"{n}-layer" for n in layer_counts])
ax.set_ylabel("Score (%)")
ax.set_ylim(95, 101)
ax.set_title("LC25000 — F1 & Accuracy vs GraphSAGE Layer Count (Fig.11b)")
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("lc25000_layer_comparison_ver2-copy.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Per-class F1 bar chart — 3-layer vs SUARA
x, width = np.arange(NUM_CLASSES), 0.35
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width/2, test_3layer["per_class_f1"]*100, width, label="3-layer (baseline)", color="#1a73e8", alpha=0.85)
ax.bar(x + width/2, test_suara["per_class_f1"]*100,  width, label="SUARA (ours)",       color="#34a853", alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES, rotation=15)
ax.set_ylabel("F1-score (%)"); ax.set_ylim(90, 101)
ax.set_title("LC25000 Per-Class F1 — AFFC Baseline vs SUARA")
ax.legend(); ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("lc25000_per_class_f1_suara-copy.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Training throughput comparison — AFFC baseline vs SUARA
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, key, ylabel, title in [
    (axes[0], "epoch_time", "Seconds per epoch", "Training time per epoch"),
    (axes[1], "throughput", "Images per second",  "Training throughput"),
]:
    baseline_vals = history_3layer[key]
    suara_vals    = history_suara[key]
    min_ep = min(len(baseline_vals), len(suara_vals))
    epochs = list(range(1, min_ep + 1))
    ax.plot(epochs, baseline_vals[:min_ep], label="3-layer baseline", color="#1a73e8", lw=2)
    ax.plot(epochs, suara_vals[:min_ep],    label="SUARA (ours)",     color="#34a853", lw=2)
    ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel)
    ax.set_title(title); ax.legend(); ax.grid(True, linestyle="--", alpha=0.4)

plt.suptitle("SUARA vs Baseline — Training Efficiency", fontsize=13)
plt.tight_layout()
plt.savefig("lc25000_suara_efficiency.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Confusion matrices — 3-layer and SUARA
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, tm, title in [
    (axes[0], test_3layer, "AFFCNet 3-layer (baseline)"),
    (axes[1], test_suara,  "AFFCNet SUARA (novelty)"),
]:
    cm = tm["confusion_matrix"]
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    ax.set_title(title); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
plt.suptitle("Confusion Matrices — LC25000 Test Set", fontsize=13)
plt.tight_layout()
plt.savefig("lc25000_confusion_matrices-copy.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# PR curves for best model and SUARA
def plot_pr_curves(y_true, y_prob, title, save_name):
    y_bin = label_binarize(y_true, classes=np.arange(NUM_CLASSES))
    plt.figure(figsize=(10, 7))
    colors = ["#1a73e8","#e8711a","#34a853","#ea4335","#9334e6"]
    for i, (cname, col) in enumerate(zip(CLASS_NAMES, colors)):
        p, r, _ = precision_recall_curve(y_bin[:, i], y_prob[:, i])
        ap = average_precision_score(y_bin[:, i], y_prob[:, i])
        plt.plot(r, p, lw=2, color=col, label=f"{cname} (AP={ap:.3f})")
    y_bin_r = y_bin.ravel(); prob_r = y_prob.ravel()
    p_m, r_m, _ = precision_recall_curve(y_bin_r, prob_r)
    micro_ap = average_precision_score(y_bin, y_prob, average="micro")
    plt.plot(r_m, p_m, "k--", lw=2.5, label=f"Micro-avg (AP={micro_ap:.3f})")
    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.title(title); plt.legend(fontsize=9)
    plt.grid(True, linestyle="--", alpha=0.4); plt.tight_layout()
    plt.savefig(save_name, dpi=150, bbox_inches="tight"); plt.show()

plot_pr_curves(test_3layer["y_true"], test_3layer["y_prob"],
               "PR Curves — AFFCNet 3-layer", "lc25000_pr_3layer.png")
plot_pr_curves(test_suara["y_true"],  test_suara["y_prob"],
               "PR Curves — AFFCNet SUARA",   "lc25000_pr_suara.png")


## Save all results

In [ ]:
torch.save({
    # Test metrics
    "test_3layer": test_3layer, "test_5layer": test_5layer,
    
    "test_nognn":  test_nognn,  "test_noaff":  test_noaff,
    "test_suara":  test_suara,
    # Histories (include epoch_time and throughput)
    "history_3layer": history_3layer, "history_5layer": history_5layer,
    
    "history_nognn":  history_nognn,  "history_noaff":  history_noaff,
    "history_suara":  history_suara,
    # Split indices
    "train_idx": train_idx, "val_idx": val_idx, "test_idx": test_idx,
    "label_map": label_map,
    "class_names": CLASS_NAMES,
}, "lc25000_affcnet_all_results-copy.pt")

print("Saved → lc25000_affcnet_all_results.pt")
print(f"\n{'Model':<28} {'F1':>8} {'Acc':>8}")
print("-" * 47)
for name, tm in [("3-layer (baseline)", test_3layer), ("SUARA (ours)", test_suara),
                 ("5-layer", test_5layer), ("no-GNN", test_nognn), ("no-AFF", test_noaff)]:
    print(f"{name:<28} {tm['weighted_f1']*100:>7.2f}%  {tm['accuracy']*100:>7.2f}%")
print(f"\nPaper target: F1=99.84%  Acc=99.84%")


In [ ]:
import json, numpy as np

def to_serializable(v):
    if isinstance(v, np.ndarray):  return v.tolist()
    if isinstance(v, torch.Tensor): return v.tolist()
    if isinstance(v, list):
        return [to_serializable(x) for x in v]
    if isinstance(v, dict):
        return {k2: to_serializable(v2) for k2, v2 in v.items()}
    return v

# Compute per-model gradient message size from largest parameter tensor
def get_max_grad_bytes(model):
    return max(p.numel() * 4 for p in model.parameters())  # float32 = 4 bytes

perf_export = {
    "ms_bytes": MS_BYTES,
    "max_grad_bytes_3layer": get_max_grad_bytes(trained_3layer),
    "max_grad_bytes_suara":  get_max_grad_bytes(trained_suara),

    # Baseline (3-layer) timing — used as "native allreduce" reference
    "baseline_epoch_times": history_3layer["epoch_time"],
    "baseline_throughput":  history_3layer["throughput"],
    "baseline_best_f1":     float(best_f1_3layer),
    "baseline_test_f1":     float(test_3layer["weighted_f1"]),
    "baseline_test_acc":    float(test_3layer["accuracy"]),
    "baseline_test_prec":   float(test_3layer["precision_w"]),
    "baseline_test_recall": float(test_3layer["recall_w"]),
    "baseline_test_spec":   float(test_3layer["specificity_macro"]),

    # SUARA-variant timing — used as "SUARA2-enhanced" reference
    "suara_epoch_times":    history_suara["epoch_time"],
    "suara_throughput":     history_suara["throughput"],
    "suara_best_f1":        float(best_f1_suara),
    "suara_test_f1":        float(test_suara["weighted_f1"]),
    "suara_test_acc":       float(test_suara["accuracy"]),
    "suara_test_prec":      float(test_suara["precision_w"]),
    "suara_test_recall":    float(test_suara["recall_w"]),
    "suara_test_spec":      float(test_suara["specificity_macro"]),

    # All ablation F1s for summary table
    "nognn_test_f1":  float(test_nognn["weighted_f1"]),
    "noaff_test_f1":  float(test_noaff["weighted_f1"]),
    "layer5_test_f1": float(test_5layer["weighted_f1"]),
}

out_path = os.path.join(HOME, "affcnet_perf_stats-copy.json")
with open(out_path, "w") as f:
    json.dump(to_serializable(perf_export), f, indent=2)

print(f"Exported performance stats → {out_path}")
print(f"  Baseline avg epoch time : {np.mean(history_3layer['epoch_time']):.2f}s")
print(f"  SUARA    avg epoch time : {np.mean(history_suara['epoch_time']):.2f}s")
print(f"  Speedup  (single GPU)   : {np.mean(history_3layer['epoch_time'])/np.mean(history_suara['epoch_time']):.2f}x")
print(f"  Baseline test F1        : {test_3layer['weighted_f1']*100:.2f}%")
print(f"  SUARA    test F1        : {test_suara['weighted_f1']*100:.2f}%")